# ⚡ Dijkstra's Algorithm: Shortest Paths in Weighted Graphs

## Learning Objectives

1. Understand weighted graphs and their biological applications
2. Implement Dijkstra's algorithm with a priority queue
3. Apply shortest path finding to confidence-weighted PPI networks

---

In [ ]:
import heapq
from collections import defaultdict

class WeightedGraph:
    """Weighted graph using adjacency list."""
    def __init__(self):
        self.adj = defaultdict(list)  # vertex -> [(neighbor, weight), ...]
    
    def add_edge(self, u, v, weight, directed=False):
        self.adj[u].append((v, weight))
        if not directed:
            self.adj[v].append((u, weight))
    
    def vertices(self):
        return list(self.adj.keys())

## 1. Dijkstra's Algorithm

Finds shortest paths from a source to all vertices in a graph with **non-negative** edge weights.

**Key idea:** Greedily select the vertex with minimum distance, update neighbors.

**Time complexity:** O((V + E) log V) with binary heap

In [ ]:
def dijkstra(graph, start):
    """
    Dijkstra's algorithm using min-heap.
    Returns: (distances, predecessors)
    """
    distances = {v: float('inf') for v in graph.vertices()}
    distances[start] = 0
    predecessors = {start: None}
    
    # Min-heap: (distance, vertex)
    heap = [(0, start)]
    visited = set()
    
    while heap:
        dist, u = heapq.heappop(heap)
        
        if u in visited:
            continue
        visited.add(u)
        
        for v, weight in graph.adj[u]:
            if v not in visited:
                new_dist = dist + weight
                if new_dist < distances[v]:
                    distances[v] = new_dist
                    predecessors[v] = u
                    heapq.heappush(heap, (new_dist, v))
    
    return distances, predecessors

def reconstruct_path(predecessors, end):
    """Reconstruct path from predecessors dictionary."""
    path = []
    current = end
    while current is not None:
        path.append(current)
        current = predecessors.get(current)
    return path[::-1]

In [ ]:
# Build weighted PPI network (weights = interaction confidence)
# Lower weight = higher confidence (we minimize distance)
ppi = WeightedGraph()
interactions = [
    ('TP53', 'MDM2', 0.01),   # Very high confidence
    ('TP53', 'BRCA1', 0.15),
    ('TP53', 'ATM', 0.08),
    ('MDM2', 'RB1', 0.20),
    ('BRCA1', 'CHEK2', 0.12),
    ('ATM', 'CHEK2', 0.05),
    ('RB1', 'E2F1', 0.10),
    ('CHEK2', 'CDC25A', 0.18),
]

for u, v, conf in interactions:
    # Convert confidence to distance: higher confidence = lower weight
    ppi.add_edge(u, v, 1 - conf)

# Find shortest paths from TP53
distances, predecessors = dijkstra(ppi, 'TP53')

print("Shortest path distances from TP53:")
for gene, dist in sorted(distances.items(), key=lambda x: x[1]):
    path = reconstruct_path(predecessors, gene)
    print(f"  {gene}: {dist:.2f} via {' → '.join(path)}")

## 🧬 Exercise: Most Reliable Path

Instead of minimizing distance, find the path that **maximizes** product of confidence scores (most reliable signaling pathway).

In [ ]:
import math

def most_reliable_path(graph, start, end, confidences):
    """
    Find path maximizing product of edge confidences.
    Hint: max(∏ conf) = max(∑ log(conf)) = min(∑ -log(conf))
    """
    # Convert to log space and use Dijkstra
    log_graph = WeightedGraph()
    for u, neighbors in graph.adj.items():
        for v, _ in neighbors:
            conf = confidences.get((u, v), confidences.get((v, u), 0.5))
            log_graph.add_edge(u, v, -math.log(conf), directed=True)
    
    distances, preds = dijkstra(log_graph, start)
    path = reconstruct_path(preds, end)
    reliability = math.exp(-distances[end])
    
    return path, reliability

# Test with original confidence scores
conf_scores = {('TP53', 'MDM2'): 0.99, ('TP53', 'BRCA1'): 0.85, ('TP53', 'ATM'): 0.92,
               ('MDM2', 'RB1'): 0.80, ('BRCA1', 'CHEK2'): 0.88, ('ATM', 'CHEK2'): 0.95,
               ('RB1', 'E2F1'): 0.90, ('CHEK2', 'CDC25A'): 0.82}

path, rel = most_reliable_path(ppi, 'TP53', 'CDC25A', conf_scores)
print(f"Most reliable path to CDC25A: {' → '.join(path)}")
print(f"Path reliability: {rel:.4f}")

---

## Summary

✅ Dijkstra finds shortest paths in weighted graphs with non-negative edges  
✅ Uses greedy selection with a priority queue  
✅ O((V + E) log V) with binary heap  
✅ Can find most reliable paths by transforming to log space

**Next:** [04_bellman_ford.ipynb](04_bellman_ford.ipynb) - Handling negative weights